# Transformación y preparación de datos para OLS — Propiedades nuevas

## 1. Alcance, carga e inspección

Única fuente de entrada: **`data/newbuilds_integrado.csv`**. No se consultan los CSV originales ni otros notebooks. El resultado será `data/newbuilds_ols.csv`, con predictores candidatos numéricos, objetivo y metadatos separados por su rol.

No se crean particiones, ni se ajustan modelos, imputadores o procedimientos de selección. Los chequeos de rango y estimabilidad de coeficientes se reservan para `Modelo_OLS.ipynb`, usando su entrenamiento. El vocabulario nominal se fija por orden lexicográfico como esquema del archivo, sin usar precios ni frecuencias para elegir categorías o referencias.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Permite ejecutar desde Propiedades_Nuevas, Ventas o la raíz del proyecto.
bases = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
candidatas_ruta = [p / "data" / "newbuilds_integrado.csv" for p in bases]
candidatas_ruta += [p / "Propiedades_Nuevas" / "data" / "newbuilds_integrado.csv" for p in bases]
candidatas_ruta += [p / "Ventas" / "Propiedades_Nuevas" / "data" / "newbuilds_integrado.csv" for p in bases]
ruta_integrado = next((p for p in candidatas_ruta if p.is_file()), None)
if ruta_integrado is None:
    raise FileNotFoundError("No se encontró Propiedades_Nuevas/data/newbuilds_integrado.csv.")

# La copia original conserva incluso las representaciones textuales de las fechas.
df_original = pd.read_csv(ruta_integrado, encoding="utf-8", float_precision="round_trip")
df_trabajo = df_original.copy(deep=True)
inspeccion_inicial = pd.DataFrame({"tipo_pandas": df_original.dtypes.astype(str),
                                   "nulos": df_original.isna().sum(),
                                   "porcentaje_nulos": df_original.isna().mean() * 100})
print("Fuente:", ruta_integrado)
print("Dimensiones originales:", df_original.shape)
display(inspeccion_inicial.rename_axis("variable").reset_index())
display(pd.Series({"filas_duplicadas": df_original.duplicated().sum(),
                   "id_duplicados": df_original.id.duplicated().sum(),
                   "id_unicos": df_original.id.nunique(),
                   "complejos_unicos": df_original.complex_id.nunique()}).to_frame("cantidad"))
assert df_original.shape == (8000, 41)
assert df_original.columns.is_unique and df_original.id.notna().all() and df_original.id.is_unique
assert not df_original.duplicated().any()
assert df_original[["id", "complex_id", "district", "okrug", "metro_station", "metro_line", "year_month"]].notna().all().all()
assert df_original[["_merge_district_prices", "_merge_metro_stations"]].eq("both").all().all()


Fuente: C:\Users\mglon\OneDrive\Desktop\UNIVERSIDAD\Introduccion al analisis de datos\MOSCOW DATASET\Moscow_EDA\Ventas\Propiedades_Nuevas\data\newbuilds_integrado.csv
Dimensiones originales: (8000, 41)


,variable,tipo_pandas,nulos,porcentaje_nulos
0,id,str,0,0.0
1,date_posted,str,0,0.0
2,complex_id,str,0,0.0
3,complex_name,str,0,0.0
4,developer,str,0,0.0
5,complex_class,str,0,0.0
6,district,str,0,0.0
7,okrug,str,0,0.0
8,lat,float64,0,0.0
9,lon,float64,0,0.0


,cantidad
filas_duplicadas,0
id_duplicados,0
id_unicos,8000
complejos_unicos,250


## 2. Fechas, integridad y objetivo

Se comprueba la coincidencia de las dos representaciones de fecha y del mes de integración. Se exige que todos los precios sean numéricos, finitos y positivos antes de obtener `ln(price_rub)`. Los registros no se descartan silenciosamente: ante una inconsistencia estructural se detiene la ejecución para revisarla.


In [2]:
df_trabajo["date_posted_dt"] = pd.to_datetime(df_trabajo.date_posted_dt, errors="raise")
fechas_publicacion = pd.to_datetime(df_trabajo.date_posted, format="%Y-%m-%d", errors="raise")
assert df_trabajo.date_posted_dt.notna().all()
pd.testing.assert_series_equal(fechas_publicacion, df_trabajo.date_posted_dt, check_names=False)
assert df_trabajo.year_month.eq(df_trabajo.date_posted_dt.dt.strftime("%Y-%m")).all()
assert pd.api.types.is_numeric_dtype(df_trabajo.price_rub)
assert df_trabajo.price_rub.notna().all() and np.isfinite(df_trabajo.price_rub).all() and df_trabajo.price_rub.gt(0).all()
df_trabajo["log_price_rub"] = np.log(df_trabajo.price_rub)
assert np.isfinite(df_trabajo.log_price_rub).all()
assert np.allclose(np.exp(df_trabajo.log_price_rub), df_trabajo.price_rub)

numericas_originales = df_original.select_dtypes(include="number").columns
assert not np.isinf(df_trabajo[numericas_originales]).any().any()
controles_fisicos = pd.Series({
    "superficie_no_positiva": df_trabajo.total_area.le(0).sum(),
    "rooms_negativos": df_trabajo.rooms.lt(0).sum(),
    "piso_no_positivo": df_trabajo.floor.le(0).sum(),
    "total_floors_no_positivo": df_trabajo.total_floors.le(0).sum(),
    "piso_mayor_al_total": df_trabajo.floor.gt(df_trabajo.total_floors).sum(),
    "tiempo_metro_negativo": df_trabajo.metro_distance_min.lt(0).sum(),
    "distancia_centro_negativa": df_trabajo.to_center_km.lt(0).sum(),
    "distancia_estacion_centro_negativa": df_trabajo.metro_to_center_km.lt(0).sum(),
    "latitud_invalida": (~df_trabajo.lat.between(-90,90)).sum(),
    "longitud_invalida": (~df_trabajo.lon.between(-180,180)).sum(),
    "latitud_metro_invalida": (~df_trabajo.metro_lat.between(-90,90)).sum(),
    "longitud_metro_invalida": (~df_trabajo.metro_lon.between(-180,180)).sum(),
})
display(controles_fisicos.to_frame("filas_a_revisar"))
assert controles_fisicos.eq(0).all(), "Revisar dominios físicos antes de transformar."
print("Estudios rooms=0 conservados:", int(df_trabajo.rooms.eq(0).sum()))
print("Periodo observado:", df_trabajo.date_posted_dt.min(), "a", df_trabajo.date_posted_dt.max())
display(df_trabajo[["price_rub", "log_price_rub"]].head())


Estudios rooms=0 conservados: 797
Periodo observado: 2021-01-01 00:00:00 a 2026-04-29 00:00:00


,filas_a_revisar
superficie_no_positiva,0
rooms_negativos,0
piso_no_positivo,0
total_floors_no_positivo,0
piso_mayor_al_total,0
tiempo_metro_negativo,0
distancia_centro_negativa,0
distancia_estacion_centro_negativa,0
latitud_invalida,0
longitud_invalida,0


,price_rub,log_price_rub
0,22075000,16.909956
1,22445000,16.926578
2,7075000,15.772078
3,3011000,14.917783
4,4420000,15.301650


## 3. Auditoría geográfica del metro

`metro_distance_min` declara minutos por su nombre, pero el CSV no incluye modo de transporte, fórmula ni documentación de unidades. Se compara con una distancia de círculo máximo calculada mediante Haversine (radio terrestre medio 6.371,0088 km). Esa distancia geográfica **no es tiempo ni longitud de un recorrido por calles**. Una asociación alta no demuestra un error de unidad.

Se calcula `assigned_metro_km` únicamente para auditar la estación asignada. También se evalúa y crea `nearest_metro_km` usando el catálogo de estaciones **observadas en este mismo CSV**, sin incorporar fuentes externas. La coordenada y año de apertura de cada estación/línea deben ser consistentes. Solo se consideran estaciones con año de apertura estrictamente anterior al año del anuncio: al no conocer día y mes de inauguración, se excluye conservadoramente el mismo año.

Este mínimo está limitado a las 104 estaciones/líneas observadas; no acredita cuál es la estación más cercana de toda Moscú. Los años de apertura disponibles son anteriores al periodo de anuncios. Se asume que sus coordenadas representan la ubicación histórica; el CSV no permite verificar cambios de ubicación o cierres. Se usa `nearest_metro_km` como candidata espacial y se reserva el tiempo de significado no verificado para auditoría, sin eliminar sus filas ni convertirlo a kilómetros.


In [3]:
columnas_catalogo = ["metro_station", "metro_line", "metro_lat", "metro_lon", "metro_year_opened"]
catalogo_metro = df_trabajo[columnas_catalogo].drop_duplicates().reset_index(drop=True)
assert not catalogo_metro.duplicated(["metro_station", "metro_line"]).any(), "Estación/línea con atributos contradictorios."
assert catalogo_metro.notna().all().all()
assert catalogo_metro.metro_year_opened.mod(1).eq(0).all()

# Distancia a la estación asignada (diagnóstico).
lat_prop = np.radians(df_trabajo.lat.to_numpy())
lon_prop = np.radians(df_trabajo.lon.to_numpy())
lat_asignada = np.radians(df_trabajo.metro_lat.to_numpy())
lon_asignada = np.radians(df_trabajo.metro_lon.to_numpy())
a_asignada = (np.sin((lat_asignada-lat_prop)/2)**2
              + np.cos(lat_prop)*np.cos(lat_asignada)*np.sin((lon_asignada-lon_prop)/2)**2)
df_trabajo["assigned_metro_km"] = 6371.0088 * 2 * np.arcsin(np.sqrt(np.clip(a_asignada,0,1)))

# Distancias de cada anuncio a las estaciones observadas; matriz 8000 x 104.
lat_estaciones = np.radians(catalogo_metro.metro_lat.to_numpy())
lon_estaciones = np.radians(catalogo_metro.metro_lon.to_numpy())
a_catalogo = (np.sin((lat_estaciones[None,:]-lat_prop[:,None])/2)**2
              + np.cos(lat_prop[:,None])*np.cos(lat_estaciones[None,:])
              * np.sin((lon_estaciones[None,:]-lon_prop[:,None])/2)**2)
distancias_catalogo = 6371.0088 * 2 * np.arcsin(np.sqrt(np.clip(a_catalogo,0,1)))
estacion_previa = catalogo_metro.metro_year_opened.to_numpy()[None,:] < df_trabajo.date_posted_dt.dt.year.to_numpy()[:,None]
distancias_historicas = np.where(estacion_previa, distancias_catalogo, np.inf)
minimos_metro = distancias_historicas.min(axis=1)
df_trabajo["nearest_metro_km"] = np.where(np.isfinite(minimos_metro), minimos_metro, np.nan)
# Si no hubiera estaciones elegibles, queda un faltante explícito, nunca un cero imputado.
print("Estaciones/líneas observadas:", len(catalogo_metro))
print("Años de apertura:", catalogo_metro.metro_year_opened.min(), "a", catalogo_metro.metro_year_opened.max())
print("Anuncios sin estación históricamente elegible:", int(df_trabajo.nearest_metro_km.isna().sum()))
display(df_trabajo[["metro_distance_min", "assigned_metro_km", "nearest_metro_km"]].describe(percentiles=[.5,.95,.99]).T)
print("Correlación descriptiva minutos/distancia asignada:", df_trabajo.metro_distance_min.corr(df_trabajo.assigned_metro_km))
display(df_trabajo.nlargest(5,"metro_distance_min")[["id","metro_distance_min","assigned_metro_km","nearest_metro_km"]])
assert df_trabajo.nearest_metro_km.dropna().ge(0).all()


Estaciones/líneas observadas: 104
Años de apertura: 1935 a 2018
Anuncios sin estación históricamente elegible: 0
Correlación descriptiva minutos/distancia asignada: 0.999923945778826


,count,mean,std,min,50%,95%,99%,max
metro_distance_min,8000.0,169.012875,187.256063,1.00000,107.000000,629.000000,815.000000,960.000000
assigned_metro_km,8000.0,13.834990,15.603715,0.03287,8.632211,52.052946,67.650696,80.004173
nearest_metro_km,8000.0,6.808109,11.673820,0.03287,1.956659,37.942673,52.843472,56.212284


,id,metro_distance_min,assigned_metro_km,nearest_metro_km
5300,N005301,960,80.004173,56.212284
5500,N005501,931,77.354776,56.212284
3919,N003920,924,77.057991,53.313284
6635,N006636,910,75.501737,47.553923
7473,N007474,909,75.801774,52.843472


## 4. Información hipotecaria, agregados mensuales y disponibilidad

`mortgage_rate_at_listing` se considera la tasa declarada al publicar, según el significado de su campo. Se comprueba su equivalencia con la tasa mensual promedio. `avg_mortgage_rate_pct` y `cbr_key_rate_pct` quedan fuera de los predictores: las tasas son idénticas o difieren por una constante y los agregados no acreditan su disponibilidad puntual.

Se excluyen también referencias de precios, variaciones y volúmenes contemporáneos. El integrado contiene únicamente los meses/distritos representados por sus anuncios, no el historial auxiliar completo ni fechas de publicación/revisión de los indicadores. Desplazar por la fila anterior de cada distrito podría saltar meses, y un mes previo tampoco acredita la fecha de difusión. Por ello no se fabrican rezagos con disponibilidad no verificable en esta etapa.

`subsidized_mortgage` se conserva como 0/1: la constancia encontrada antes en un entrenamiento específico no justifica excluirla de este archivo general. La comprobación de estimabilidad dependerá del entrenamiento futuro.


In [4]:
tasas_hipotecarias = ["mortgage_rate_at_listing", "avg_mortgage_rate_pct", "cbr_key_rate_pct"]
display(df_trabajo[tasas_hipotecarias].corr())
display(pd.DataFrame({
    "avg_menos_listing": df_trabajo.avg_mortgage_rate_pct-df_trabajo.mortgage_rate_at_listing,
    "avg_menos_cbr": df_trabajo.avg_mortgage_rate_pct-df_trabajo.cbr_key_rate_pct
}).describe().T)
print("Tasas anuncio y promedio idénticas:", df_trabajo.mortgage_rate_at_listing.equals(df_trabajo.avg_mortgage_rate_pct))
assert pd.api.types.is_bool_dtype(df_trabajo.subsidized_mortgage)
assert df_trabajo.subsidized_mortgage.notna().all()
df_trabajo["subsidized_mortgage"] = df_trabajo.subsidized_mortgage.astype("int64")
display(df_trabajo.subsidized_mortgage.value_counts(dropna=False).to_frame("anuncios"))

agregados_mensuales = ["secondary_price_per_sqm", "newbuild_price_per_sqm", "secondary_mom_change_pct",
                       "newbuild_mom_change_pct", "n_listings_secondary", "n_listings_newbuild",
                       "cbr_key_rate_pct", "avg_mortgage_rate_pct"]
claves_contexto = ["year_month", "district", "okrug"]
contexto_observado = df_trabajo[claves_contexto+agregados_mensuales].drop_duplicates()
assert not contexto_observado.duplicated(claves_contexto).any(), "Valores mensuales contradictorios para una clave."
print("Claves mensuales observadas (historia parcial):", len(contexto_observado))


Tasas anuncio y promedio idénticas: True
Claves mensuales observadas (historia parcial): 4425


,mortgage_rate_at_listing,avg_mortgage_rate_pct,cbr_key_rate_pct
mortgage_rate_at_listing,1.0,1.0,1.0
avg_mortgage_rate_pct,1.0,1.0,1.0
cbr_key_rate_pct,1.0,1.0,1.0


,count,mean,std,min,25%,50%,75%,max
avg_menos_listing,8000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
avg_menos_cbr,8000.0,2.5,0.0,2.5,2.5,2.5,2.5,2.5


,anuncios
subsidized_mortgage,
1,5247
0,2753


## 5. Variables temporales y posición de la unidad

Se derivan `listing_year` y `listing_month`. El mes entra en los predictores como seno y coseno de periodo 12 (enero: ángulo cero), evitando tratar diciembre y enero como extremos lejanos.

`months_to_completion = 12*(completion_year-listing_year)+12-listing_month` aproxima entrega a **diciembre del año declarado**, sin precisión de día. Si la entrega real ocurre antes de diciembre, la aproximación puede excederla hasta 11 meses. Se conservan negativos. `completion_year` y el mes crudo se reservan fuera de los predictores, para no añadir representaciones temporales redundantes. No se crea antigüedad del edificio.

`relative_floor = floor/total_floors`; los dos pisos originales también siguen siendo candidatos. No es una combinación lineal exacta de ambos.

La interpretación predictiva de `ready_status` y `completion_year` exige que sean el estado y la entrega prevista **declarados al publicar**. El CSV no contiene versiones históricas que lo prueben: quedan como candidatas condicionadas a esa interpretación, no como variables cuya disponibilidad se haya demostrado. Se informan aparentes contradicciones para su revisión, sin cambiar estados o fechas automáticamente. El notebook de modelado deberá confirmar esta procedencia o excluir `ready_status` y `months_to_completion` del baseline histórico.


In [5]:
assert df_trabajo.completion_year.dropna().mod(1).eq(0).all()
df_trabajo["listing_year"] = df_trabajo.date_posted_dt.dt.year
df_trabajo["listing_month"] = df_trabajo.date_posted_dt.dt.month
df_trabajo["listing_month_sin"] = np.sin(2*np.pi*(df_trabajo.listing_month-1)/12)
df_trabajo["listing_month_cos"] = np.cos(2*np.pi*(df_trabajo.listing_month-1)/12)
df_trabajo["months_to_completion"] = 12*(df_trabajo.completion_year-df_trabajo.listing_year)+12-df_trabajo.listing_month
df_trabajo["relative_floor"] = df_trabajo.floor/df_trabajo.total_floors
auditoria_temporal = pd.Series({
    "plazo_aproximado_negativo_conservado": df_trabajo.months_to_completion.lt(0).sum(),
    "completed_con_anio_entrega_posterior_al_anuncio": (df_trabajo.ready_status.eq("completed") & df_trabajo.completion_year.gt(df_trabajo.listing_year)).sum(),
    "under_construction_con_anio_entrega_pasado": (df_trabajo.ready_status.eq("under_construction") & df_trabajo.completion_year.lt(df_trabajo.listing_year)).sum(),
    "estacion_asignada_abierta_en_anio_del_anuncio_o_despues": df_trabajo.metro_year_opened.ge(df_trabajo.listing_year).sum(),
})
display(auditoria_temporal.to_frame("filas_a_revisar"))
display(df_trabajo[["date_posted_dt","completion_year","listing_year","listing_month","listing_month_sin","listing_month_cos","months_to_completion","relative_floor"]].head())
assert "building_age_at_listing" not in df_trabajo


,filas_a_revisar
plazo_aproximado_negativo_conservado,315
completed_con_anio_entrega_posterior_al_anuncio,0
under_construction_con_anio_entrega_pasado,0
estacion_asignada_abierta_en_anio_del_anuncio_o_despues,0


,date_posted_dt,completion_year,listing_year,listing_month,listing_month_sin,listing_month_cos,months_to_completion,relative_floor
0,2023-11-22,2027,2023,11,-8.660254e-01,0.500000,49,0.714286
1,2021-07-08,2025,2021,7,1.224647e-16,-1.000000,53,0.647059
2,2022-02-20,2025,2022,2,5.000000e-01,0.866025,46,0.800000
3,2025-03-23,2027,2025,3,8.660254e-01,0.500000,33,0.500000
4,2022-08-05,2026,2022,8,-5.000000e-01,-0.866025,52,1.000000


## 6. Encoding determinista de las seis nominales

Se fija el vocabulario mediante valores únicos ordenados lexicográficamente, sin estadísticas de frecuencia ni decisiones basadas en el precio. Es un esquema de serialización para este archivo completo, **no un codificador aprendido en un conjunto de entrenamiento**. La primera categoría se omite como referencia; se registra cada nivel y el número de dummies. No se agrupan categorías raras.

Si aparecen faltantes categóricos se conservan como tales antes del encoding y se marcan todas las dummies de su bloque como `NaN`, evitando confundirlos con la referencia. En estos datos no hay faltantes. En el modelado, los niveles ausentes en entrenamiento, las columnas constantes y las categorías nuevas deberán tratarse usando únicamente entrenamiento, sin reutilizar frecuencias de todo el archivo. No se incorpora `okrug` junto con `district`, ni `metro_station` como nominal.


In [6]:
categoricas_ols = ["district","developer","complex_class","ready_status","deal_type","metro_line"]
bloques_dummies = []
reporte_encoding = []
columnas_por_categoria = {}
for columna in categoricas_ols:
    niveles = sorted(df_trabajo[columna].dropna().unique().tolist())
    assert niveles, f"{columna} no tiene categorías observadas."
    serie = pd.Categorical(df_trabajo[columna], categories=niveles, ordered=False)
    bloque = pd.get_dummies(serie, prefix=columna, drop_first=True, dtype="float64")
    bloque.index = df_trabajo.index
    bloque.loc[df_trabajo[columna].isna(), :] = np.nan
    bloques_dummies.append(bloque)
    columnas_por_categoria[columna] = bloque.columns.tolist()
    reporte_encoding.append({"variable":columna, "referencia":niveles[0], "categorias":len(niveles),
                              "columnas_generadas":bloque.shape[1], "niveles_del_esquema":", ".join(niveles)})
dummies_ols = pd.concat(bloques_dummies, axis=1)
reporte_encoding = pd.DataFrame(reporte_encoding)
display(reporte_encoding)
assert dummies_ols.columns.is_unique
assert dummies_ols.shape[1] == reporte_encoding.columnas_generadas.sum()


,variable,referencia,categorias,columnas_generadas,niveles_del_esquema
0,district,Akademichesky,109,108,"Akademichesky, Alekseevsky, Altufyevsky, Arbat..."
1,developer,Capital Group,11,10,"Capital Group, DONSTROY, Etalon, FSK, GK A101,..."
2,complex_class,business,4,3,"business, comfort, economy, premium"
3,ready_status,completed,2,1,"completed, under_construction"
4,deal_type,DDU,4,3,"DDU, FZ214, installment, trade-in"
5,metro_line,Arbatsko-Pokrovskaya,12,11,"Arbatsko-Pokrovskaya, Bolshaya Koltsevaya, Fil..."


## 7. Predictores candidatos, faltantes y comprobaciones del archivo

Las numéricas conservan sus escalas originales; no se estandariza ni se agregan logaritmos, polinomios o interacciones. Se incluyen la variable subsidiada y las características físicas, espaciales y temporales indicadas. `assigned_metro_km` queda solo como diagnóstico; `nearest_metro_km` es la alternativa espacial de alcance documentado.

Se informan faltantes antes y después de transformar, constantes y columnas con valores idénticos. Ninguna columna se retira por constancia en una partición anterior. Si existieran faltantes, se mantendrían para una imputación ajustada exclusivamente en entrenamiento más adelante. En esta ejecución no hay valores pendientes. La ausencia de columnas duplicadas por valores se comprueba; el rango con intercepto y la estimabilidad de coeficientes quedan expresamente para el modelado.


In [7]:
numericas_ols = ["total_area","rooms","floor","total_floors","to_center_km","lat","lon",
                  "mortgage_rate_at_listing","metro_lat","metro_lon","metro_year_opened","metro_to_center_km",
                  "subsidized_mortgage","listing_year","listing_month_sin","listing_month_cos",
                  "months_to_completion","relative_floor","nearest_metro_km"]
columnas_entrada = numericas_ols+categoricas_ols
faltantes_antes = pd.DataFrame({"nulos":df_trabajo[columnas_entrada].isna().sum(),
                                "porcentaje":df_trabajo[columnas_entrada].isna().mean()*100})
display(faltantes_antes.rename_axis("variable").reset_index())
# Solo conversión de tipo: no imputación ni estimación de parámetros.
X_candidatas = pd.concat([df_trabajo[numericas_ols].astype("float64"), dummies_ols], axis=1)
columnas_predictoras = X_candidatas.columns.tolist()
assert X_candidatas.columns.is_unique
assert all(pd.api.types.is_numeric_dtype(t) and not pd.api.types.is_bool_dtype(t) for t in X_candidatas.dtypes)
assert not np.isinf(X_candidatas.to_numpy()).any()
constantes_archivo = X_candidatas.columns[X_candidatas.nunique(dropna=False).le(1)].tolist()
duplicadas_archivo = X_candidatas.columns[X_candidatas.T.duplicated(keep=False)].tolist()
display(pd.DataFrame({"columna_constante":constantes_archivo}))
display(pd.DataFrame({"columna_duplicada_por_valores":duplicadas_archivo}))
assert not duplicadas_archivo, "Revisar equivalencia de columnas antes de exportar."
faltantes_pendientes = X_candidatas.isna().sum()
display(faltantes_pendientes[faltantes_pendientes.gt(0)].to_frame("nulos_pendientes"))
print("Nulos pendientes totales:", int(faltantes_pendientes.sum()))
print("Dimensiones de predictores candidatos:", X_candidatas.shape)


Nulos pendientes totales: 0
Dimensiones de predictores candidatos: (8000, 155)


,variable,nulos,porcentaje
0,total_area,0,0.0
1,rooms,0,0.0
2,floor,0,0.0
3,total_floors,0,0.0
4,to_center_km,0,0.0
5,lat,0,0.0
6,lon,0,0.0
7,mortgage_rate_at_listing,0,0.0
8,metro_lat,0,0.0
9,metro_lon,0,0.0


,columna_constante


,columna_duplicada_por_valores


,nulos_pendientes


## 8. Dataset transformado, metadatos y exportación

Se exportan los predictores y cinco campos de rol reservado: `log_price_rub` (objetivo), `price_rub` (evaluación futura en RUB), `id`, `complex_id` y `date_posted_dt` (metadatos). El notebook de modelado debe separarlos de X antes de ajustar OLS.

No se agregan indicadores de entrenamiento, validación o prueba. Se verifican recarga, nombres, orden, valores, identificadores y fechas; no se modifican la fuente integrada ni el EDA.


In [8]:
columnas_reservadas = ["log_price_rub","price_rub","id","complex_id","date_posted_dt"]
assert not set(columnas_reservadas).intersection(columnas_predictoras)
df_ols = pd.concat([df_trabajo[columnas_reservadas].copy(),X_candidatas],axis=1)
assert df_ols.columns.is_unique
assert len(df_ols) == len(df_original) == 8000
assert df_ols.id.is_unique and df_ols.id.eq(df_original.id).all()
assert df_ols.complex_id.eq(df_original.complex_id).all()
assert df_ols.price_rub.eq(df_original.price_rub).all()
assert df_ols.shape[1] == len(numericas_ols)+reporte_encoding.columnas_generadas.sum()+len(columnas_reservadas)
assert "subsidized_mortgage" in columnas_predictoras
assert df_ols.subsidized_mortgage.isin([0,1]).all()
assert "metro_distance_min" not in columnas_predictoras
assert not set(["price_per_sqm","newbuild_price_per_sqm","okrug","metro_station"]+agregados_mensuales).intersection(columnas_predictoras)

ruta_ols = ruta_integrado.parent / "newbuilds_ols.csv"
df_ols.to_csv(ruta_ols,index=False,encoding="utf-8")
df_ols_recargado = pd.read_csv(ruta_ols,encoding="utf-8",parse_dates=["date_posted_dt"],float_precision="round_trip")
pd.testing.assert_frame_equal(df_ols_recargado,df_ols,check_dtype=False,check_exact=True)
assert pd.api.types.is_datetime64_any_dtype(df_ols_recargado.date_posted_dt)
# La copia de entrada permanece sin mutaciones.
pd.testing.assert_frame_equal(df_original,pd.read_csv(ruta_integrado,encoding="utf-8",float_precision="round_trip"),check_exact=True)
print("Exportado y recargado sin cambios de valores:",ruta_ols)
print("Dimensiones del CSV:",df_ols_recargado.shape)
print("En Modelo_OLS: y = df_ols['log_price_rub']; X = df_ols.drop(columns=columnas_reservadas).")


Exportado y recargado sin cambios de valores: C:\Users\mglon\OneDrive\Desktop\UNIVERSIDAD\Introduccion al analisis de datos\MOSCOW DATASET\Moscow_EDA\Ventas\Propiedades_Nuevas\data\newbuilds_ols.csv
Dimensiones del CSV: (8000, 160)
En Modelo_OLS: y = df_ols['log_price_rub']; X = df_ols.drop(columns=columnas_reservadas).


## Resumen de transformaciones para OLS

Una característica conceptual puede producir varias columnas: las seis nominales generan K−1 dummies por categoría; los campos temporales originan distintas representaciones. Los conteos no deben confundirse con el número de columnas del CSV, que también contiene objetivo y metadatos.

El cuadro siguiente documenta todas las variables originales, incluidas las excluidas, y las derivadas. El esquema de categorías y sus referencias queda en la sección 6; no se crea ningún archivo auxiliar.


In [9]:
motivos_exclusion = {
    "complex_name":"Etiqueta del complejo; solo se conserva complex_id como metadato.",
    "price_per_sqm":"Derivada del precio: fuga directa de información.",
    "okrug":"Jerarquía geográfica redundante con district en el diseño inicial.",
    "metro_station":"No se incorpora como nominal; alta cardinalidad. Se usa para el catálogo geográfico de auditoría.",
    "metro_distance_min":"Tiempo declarado sin método/unidad verificables; no se convierte ni se eliminan observaciones. Se propone nearest_metro_km.",
    "date_posted":"Texto original redundante; se conserva date_posted_dt como metadato.",
    "year_month":"Clave del join, no predictor; el calendario tiene representaciones numéricas.",
    "completion_year":"Insumo para months_to_completion; fuera de X para evitar redundancias temporales.",
    "listing_month":"Insumo para seno/coseno; fuera de X en forma cruda.",
    "assigned_metro_km":"Diagnóstico geográfico de la estación asignada; no se suma como predictor adicional.",
}
for c in agregados_mensuales:
    motivos_exclusion[c] = "Agregado contemporáneo sin fecha de disponibilidad acreditada; no se construyen rezagos con historia parcial."
motivos_exclusion["newbuild_price_per_sqm"] += " Precio del mismo mercado/distrito/mes: posible fuga."
for c in ["avg_mortgage_rate_pct","cbr_key_rate_pct"]:
    motivos_exclusion[c] += " Además, equivalencia exacta o afín con mortgage_rate_at_listing."
for c in df_original.columns:
    if c.startswith("_merge_"):
        motivos_exclusion[c] = "Control de integración, fuera de los predictores y del CSV de modelado."

formulas_derivadas = {
    "log_price_rub":("ln(price_rub)","Objetivo OLS; precio finito y positivo."),
    "listing_year":("Año de date_posted_dt","Tendencia anual interpretable."),
    "listing_month_sin":("sin(2π*(listing_month-1)/12)","Calendario cíclico."),
    "listing_month_cos":("cos(2π*(listing_month-1)/12)","Calendario cíclico."),
    "months_to_completion":("12*(completion_year-listing_year)+12-listing_month","Entrega aproximada a diciembre; negativos conservados, disponibilidad histórica pendiente."),
    "relative_floor":("floor/total_floors","Posición relativa; se conservan pisos originales."),
    "nearest_metro_km":("Mínimo Haversine entre estaciones observadas con apertura anterior al año del anuncio","Kilómetros geográficos; catálogo parcial, no tiempo de viaje."),
}
filas_resumen = []
for c in df_trabajo.columns:
    if c in categoricas_ols:
        transformacion = "One-Hot Encoding determinista; primera categoría lexicográfica como referencia"
        resultado = ", ".join(columnas_por_categoria[c])
        justificacion = "Nominal sin orden ni agrupación por precio/frecuencia."
        if c == "ready_status": justificacion += " Candidata condicionada a verificar estado al publicar."
    elif c in motivos_exclusion:
        transformacion,resultado,justificacion = "Reservada para auditoría; no ingresa al CSV de modelado","Fuera de X",motivos_exclusion[c]
        if c == "completion_year": resultado = "months_to_completion"
        if c in ["date_posted","listing_month"]: resultado = "listing_year, listing_month_sin, listing_month_cos" if c=="date_posted" else "listing_month_sin, listing_month_cos"
    elif c in formulas_derivadas:
        transformacion,justificacion = formulas_derivadas[c]
        resultado = c
    elif c in columnas_reservadas:
        transformacion,resultado,justificacion = "Conservar sin uso como predictor",c,"Metadato de trazabilidad/fecha; price_rub se reserva para evaluación en moneda original."
    elif c == "subsidized_mortgage":
        transformacion,resultado,justificacion = "True→1; False→0",c,"Binaria válida, conservada sin depender de una partición anterior."
    else:
        assert c in numericas_ols, f"Variable sin documentar: {c}"
        transformacion,resultado,justificacion = "Numérica en su escala original",c,"Sin imputación global, recorte ni transformación adicional."
    filas_resumen.append({"Variable original":c,"Transformación aplicada":transformacion,"Variable resultante":resultado,"Justificación":justificacion})
resumen_transformaciones = pd.DataFrame(filas_resumen)
tabla_resumen = "| "+" | ".join(resumen_transformaciones.columns)+" |\n| --- | --- | --- | --- |\n"
for fila in resumen_transformaciones.itertuples(index=False,name=None):
    tabla_resumen += "| "+" | ".join(str(v).replace("|","\\|").replace("\n"," ") for v in fila)+" |\n"
print("Tabla Markdown regenerable con display(Markdown(tabla_resumen)).")


Tabla Markdown regenerable con display(Markdown(tabla_resumen)).


| Variable original | Transformación aplicada | Variable resultante | Justificación |
| --- | --- | --- | --- |
| id | Conservar sin uso como predictor | id | Metadato de trazabilidad/fecha; price_rub se reserva para evaluación en moneda original. |
| date_posted | Reservada para auditoría; no ingresa al CSV de modelado | listing_year, listing_month_sin, listing_month_cos | Texto original redundante; se conserva date_posted_dt como metadato. |
| complex_id | Conservar sin uso como predictor | complex_id | Metadato de trazabilidad/fecha; price_rub se reserva para evaluación en moneda original. |
| complex_name | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | Etiqueta del complejo; solo se conserva complex_id como metadato. |
| developer | One-Hot Encoding determinista; primera categoría lexicográfica como referencia | developer_DONSTROY, developer_Etalon, developer_FSK, developer_GK A101, developer_GK Granel, developer_INGRAD, developer_LSR, developer_MR Group, developer_PIK, developer_Samolet | Nominal sin orden ni agrupación por precio/frecuencia. |
| complex_class | One-Hot Encoding determinista; primera categoría lexicográfica como referencia | complex_class_comfort, complex_class_economy, complex_class_premium | Nominal sin orden ni agrupación por precio/frecuencia. |
| district | One-Hot Encoding determinista; primera categoría lexicográfica como referencia | district_Alekseevsky, district_Altufyevsky, district_Arbat, district_Babushkinsky, district_Basmanny, district_Begovoy, district_Beskudnikovsky, district_Bibirevo, district_Biryulyovo Zapadnoye, district_Bogorodskoye, district_Brateyevo, district_Chertanovo Tsentralnoye, district_Cheryomushki, district_Danilovsky, district_Desyonovskoye, district_Dmitrovsky, district_Donskoy, district_Dorogomilovo, district_Fili-Davydkovo, district_Filimonkovskoye, district_Gagarinsky, district_Golovinsky, district_Ivanovskoye, district_Izmaylovo, district_Khamovniki, district_Khoroshyovo-Mnyovniki, district_Khoroshyovsky, district_Khovrino, district_Kievsky, district_Klyonovskoye, district_Konkovo, district_Koptevo, district_Kosino-Ukhtomsky, district_Krasnopakhorskoye, district_Krylatskoye, district_Kuntsevo, district_Kurkino, district_Levoberezhny, district_Lomonosovsky, district_Losinoostrovsky, district_Marfino, district_Marushkinskoye, district_Maryina Roshcha, district_Meshchansky, district_Metrogorodok, district_Mikhaylovo-Yartsevskoye, district_Mitino, district_Molzhaninovsky, district_Moskovsky, district_Moskvorechye-Saburovo, district_Mosrentgen, district_Nagatino-Sadovniki, district_Nagatinsky Zaton, district_Novo-Peredelkino, district_Novofyodorovskoye, district_Novogireyevo, district_Novokosino, district_Obruchevsky, district_Ochakovo-Matveyevskoye, district_Orekhovo-Borisovo Severnoye, district_Orekhovo-Borisovo Yuzhnoye, district_Ostankinsky, district_Perovo, district_Pervomayskoye, district_Pokrovskoye-Streshnevo, district_Preobrazhenskoye, district_Presnensky, district_Rogovskoye, district_Rostokino, district_Ryazanovskoye, district_Savyolovsky, district_Severnoye Butovo, district_Severnoye Izmaylovo, district_Severnoye Medvedkovo, district_Severnoye Tushino, district_Severny, district_Shchapovskoye, district_Shcherbinka, district_Shchukino, district_Sokol, district_Sokolinaya Gora, district_Sokolniki, district_Solntsevo, district_Sosenskoye, district_Strogino, district_Sviblovo, district_Timiryazevsky, district_Troparyovo-Nikulino, district_Tverskoy, district_Veshnyaki, district_Vnukovo, district_Vnukovskoye, district_Voronovskoye, district_Voskresenskoye, district_Vostochnoye Izmaylovo, district_Vostochny, district_Voykovsky, district_Yakimanka, district_Yaroslavsky, district_Yasenevo, district_Yuzhnoye Butovo, district_Yuzhnoye Medvedkovo, district_Yuzhnoye Tushino, district_Zamoskvorechye, district_Zapadnoye Degunino, district_Zelenograd, district_Zyablikovo, district_Zyuzino | Nominal sin orden ni agrupación por precio/frecuencia. |
| okrug | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | Jerarquía geográfica redundante con district en el diseño inicial. |
| lat | Numérica en su escala original | lat | Sin imputación global, recorte ni transformación adicional. |
| lon | Numérica en su escala original | lon | Sin imputación global, recorte ni transformación adicional. |
| total_area | Numérica en su escala original | total_area | Sin imputación global, recorte ni transformación adicional. |
| rooms | Numérica en su escala original | rooms | Sin imputación global, recorte ni transformación adicional. |
| floor | Numérica en su escala original | floor | Sin imputación global, recorte ni transformación adicional. |
| total_floors | Numérica en su escala original | total_floors | Sin imputación global, recorte ni transformación adicional. |
| completion_year | Reservada para auditoría; no ingresa al CSV de modelado | months_to_completion | Insumo para months_to_completion; fuera de X para evitar redundancias temporales. |
| ready_status | One-Hot Encoding determinista; primera categoría lexicográfica como referencia | ready_status_under_construction | Nominal sin orden ni agrupación por precio/frecuencia. Candidata condicionada a verificar estado al publicar. |
| metro_station | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | No se incorpora como nominal; alta cardinalidad. Se usa para el catálogo geográfico de auditoría. |
| metro_line | One-Hot Encoding determinista; primera categoría lexicográfica como referencia | metro_line_Bolshaya Koltsevaya, metro_line_Filyovskaya, metro_line_Kalininskaya, metro_line_Kaluzhsko-Rizhskaya, metro_line_Koltsevaya, metro_line_Lyublinsko-Dmitrovskaya, metro_line_MCC, metro_line_Serpukhovsko-Timiryazevskaya, metro_line_Sokolnicheskaya, metro_line_Tagansko-Krasnopresnenskaya, metro_line_Zamoskvoretskaya | Nominal sin orden ni agrupación por precio/frecuencia. |
| metro_distance_min | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | Tiempo declarado sin método/unidad verificables; no se convierte ni se eliminan observaciones. Se propone nearest_metro_km. |
| to_center_km | Numérica en su escala original | to_center_km | Sin imputación global, recorte ni transformación adicional. |
| price_rub | Conservar sin uso como predictor | price_rub | Metadato de trazabilidad/fecha; price_rub se reserva para evaluación en moneda original. |
| price_per_sqm | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | Derivada del precio: fuga directa de información. |
| deal_type | One-Hot Encoding determinista; primera categoría lexicográfica como referencia | deal_type_FZ214, deal_type_installment, deal_type_trade-in | Nominal sin orden ni agrupación por precio/frecuencia. |
| subsidized_mortgage | True→1; False→0 | subsidized_mortgage | Binaria válida, conservada sin depender de una partición anterior. |
| mortgage_rate_at_listing | Numérica en su escala original | mortgage_rate_at_listing | Sin imputación global, recorte ni transformación adicional. |
| date_posted_dt | Conservar sin uso como predictor | date_posted_dt | Metadato de trazabilidad/fecha; price_rub se reserva para evaluación en moneda original. |
| year_month | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | Clave del join, no predictor; el calendario tiene representaciones numéricas. |
| secondary_price_per_sqm | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | Agregado contemporáneo sin fecha de disponibilidad acreditada; no se construyen rezagos con historia parcial. |
| newbuild_price_per_sqm | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | Agregado contemporáneo sin fecha de disponibilidad acreditada; no se construyen rezagos con historia parcial. Precio del mismo mercado/distrito/mes: posible fuga. |
| secondary_mom_change_pct | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | Agregado contemporáneo sin fecha de disponibilidad acreditada; no se construyen rezagos con historia parcial. |
| newbuild_mom_change_pct | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | Agregado contemporáneo sin fecha de disponibilidad acreditada; no se construyen rezagos con historia parcial. |
| n_listings_secondary | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | Agregado contemporáneo sin fecha de disponibilidad acreditada; no se construyen rezagos con historia parcial. |
| n_listings_newbuild | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | Agregado contemporáneo sin fecha de disponibilidad acreditada; no se construyen rezagos con historia parcial. |
| cbr_key_rate_pct | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | Agregado contemporáneo sin fecha de disponibilidad acreditada; no se construyen rezagos con historia parcial. Además, equivalencia exacta o afín con mortgage_rate_at_listing. |
| avg_mortgage_rate_pct | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | Agregado contemporáneo sin fecha de disponibilidad acreditada; no se construyen rezagos con historia parcial. Además, equivalencia exacta o afín con mortgage_rate_at_listing. |
| _merge_district_prices | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | Control de integración, fuera de los predictores y del CSV de modelado. |
| metro_lat | Numérica en su escala original | metro_lat | Sin imputación global, recorte ni transformación adicional. |
| metro_lon | Numérica en su escala original | metro_lon | Sin imputación global, recorte ni transformación adicional. |
| metro_year_opened | Numérica en su escala original | metro_year_opened | Sin imputación global, recorte ni transformación adicional. |
| metro_to_center_km | Numérica en su escala original | metro_to_center_km | Sin imputación global, recorte ni transformación adicional. |
| _merge_metro_stations | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | Control de integración, fuera de los predictores y del CSV de modelado. |
| log_price_rub | ln(price_rub) | log_price_rub | Objetivo OLS; precio finito y positivo. |
| assigned_metro_km | Reservada para auditoría; no ingresa al CSV de modelado | Fuera de X | Diagnóstico geográfico de la estación asignada; no se suma como predictor adicional. |
| nearest_metro_km | Mínimo Haversine entre estaciones observadas con apertura anterior al año del anuncio | nearest_metro_km | Kilómetros geográficos; catálogo parcial, no tiempo de viaje. |
| listing_year | Año de date_posted_dt | listing_year | Tendencia anual interpretable. |
| listing_month | Reservada para auditoría; no ingresa al CSV de modelado | listing_month_sin, listing_month_cos | Insumo para seno/coseno; fuera de X en forma cruda. |
| listing_month_sin | sin(2π*(listing_month-1)/12) | listing_month_sin | Calendario cíclico. |
| listing_month_cos | cos(2π*(listing_month-1)/12) | listing_month_cos | Calendario cíclico. |
| months_to_completion | 12*(completion_year-listing_year)+12-listing_month | months_to_completion | Entrega aproximada a diciembre; negativos conservados, disponibilidad histórica pendiente. |
| relative_floor | floor/total_floors | relative_floor | Posición relativa; se conservan pisos originales. |


In [10]:
resumen_final = pd.Series({
    "observaciones_procesadas_y_conservadas":len(df_ols),
    "variables_originales_inspeccionadas":df_original.shape[1],
    "caracteristicas_originales_que_aportan_a_X":len(set([c for c in numericas_ols if c in df_original]+categoricas_ols+["date_posted_dt","completion_year"])),
    "caracteristicas_entrada_incluidas_derivadas":len(columnas_entrada),
    "columnas_generadas_encoding":dummies_ols.shape[1],
    "columnas_numericas_predictoras":len(columnas_predictoras),
    "objetivo_precio_y_metadatos":len(columnas_reservadas),
    "columnas_CSV_exportado":df_ols.shape[1],
    "faltantes_pendientes_predictoras":int(X_candidatas.isna().sum().sum()),
    "faltantes_pendientes_CSV":int(df_ols.isna().sum().sum()),
    "columnas_constantes_en_archivo":len(constantes_archivo),
    "columnas_duplicadas_en_archivo":len(duplicadas_archivo),
    "observaciones_eliminadas":len(df_original)-len(df_ols),
},name="cantidad")
display(resumen_final.to_frame())
display(pd.DataFrame([{"variable":c,"motivo":m} for c,m in motivos_exclusion.items()]))
print("Archivo final:",ruta_ols,"| dimensiones:",df_ols.shape)
print("Roles reservados (fuera de X):",columnas_reservadas)
print("Sin entrenamiento, evaluación, selección, particiones o imputación estadística.")


Archivo final: C:\Users\mglon\OneDrive\Desktop\UNIVERSIDAD\Introduccion al analisis de datos\MOSCOW DATASET\Moscow_EDA\Ventas\Propiedades_Nuevas\data\newbuilds_ols.csv | dimensiones: (8000, 160)
Roles reservados (fuera de X): ['log_price_rub', 'price_rub', 'id', 'complex_id', 'date_posted_dt']
Sin entrenamiento, evaluación, selección, particiones o imputación estadística.


,cantidad
observaciones_procesadas_y_conservadas,8000
variables_originales_inspeccionadas,41
caracteristicas_originales_que_aportan_a_X,21
caracteristicas_entrada_incluidas_derivadas,25
columnas_generadas_encoding,136
columnas_numericas_predictoras,155
objetivo_precio_y_metadatos,5
columnas_CSV_exportado,160
faltantes_pendientes_predictoras,0
faltantes_pendientes_CSV,0


,variable,motivo
0,complex_name,Etiqueta del complejo; solo se conserva comple...
1,price_per_sqm,Derivada del precio: fuga directa de información.
2,okrug,Jerarquía geográfica redundante con district e...
3,metro_station,No se incorpora como nominal; alta cardinalida...
4,metro_distance_min,Tiempo declarado sin método/unidad verificable...
5,date_posted,Texto original redundante; se conserva date_po...
6,year_month,"Clave del join, no predictor; el calendario ti..."
7,completion_year,Insumo para months_to_completion; fuera de X p...
8,listing_month,Insumo para seno/coseno; fuera de X en forma c...
9,assigned_metro_km,Diagnóstico geográfico de la estación asignada...


Las correcciones aplicadas son estructurales: tasa financiera única, hipoteca subsidiada binaria conservada, calendario cíclico, plazo aproximado documentado, piso relativo y distancia geográfica de alcance limitado. No se corrigieron automáticamente unidades o fechas que el CSV no permite verificar.

Persisten incertidumbres de procedencia: tiempo al metro, cobertura del catálogo de estaciones y versiones históricas del estado/entrega. La correlación entre minutos y distancia asignada es aproximadamente 0,999924; describe la relación, no demuestra equivalencia de unidades. Las alertas temporales no justifican modificar valores sin evidencia. El CSV es un conjunto de **candidatas transformadas**, listo para que `Modelo_OLS.ipynb` defina particiones, confirme disponibilidad de atributos y construya matrices definitivas, incluyendo tratamiento de niveles no estimables y chequeo de rango con intercepto solo en entrenamiento.
